
# Government Executive Summary
## Predictive Forecasting of Care Load & Placement Demand

**Purpose:** Provide government decision-makers and operational leaders
with an evidence-based overview of historical care demand, forecasted
demand, uncertainty, surge risks, capacity pressures, and operational
recommendations.

**Primary forecasting target:** Children in HHS care.

**Forecast horizons:** 1, 7, 14, and 30 days.

**Evidence policy:** All numerical findings must come from the project's
dataset and generated forecasting outputs. Missing evidence must be
identified explicitly rather than replaced with invented estimates.

---

## Report Sections

1. Executive overview
2. Historical care demand
3. Forecast horizons
4. Forecast accuracy
5. Surge warnings
6. Capacity breach risks
7. Scenario comparison
8. Operational implications
9. Government recommendations
10. Limitations and uncertainty
11. Evidence inventory
12. Export to DOCX and PDF


## Evidence Analysis
Run the following cell to inspect historical demand, forecast outputs, accuracy, risk files, scenarios, and uncertainty.

In [1]:

# Analyze Historical Demand, Forecasts, Risks, and Scenarios

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR = PROJECT_ROOT / "government_executive_summary"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load the cleaned historical dataset
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

TARGET = "children_in_hhs_care"

if TARGET not in df.columns:
    raise KeyError(f"Primary target column not found: {TARGET}")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
historical = df.dropna(subset=[TARGET]).copy()

if historical.empty:
    raise ValueError("No valid historical observations are available.")

# Historical demand indicators
historical_summary = pd.DataFrame({
    "Indicator": [
        "First observed date",
        "Last observed date",
        "Number of valid observations",
        "Average children in HHS care",
        "Median children in HHS care",
        "Minimum observed care load",
        "Maximum observed care load",
        "Latest observed care load",
        "Standard deviation",
        "First-to-last change"
    ],
    "Value": [
        historical["date"].min().date(),
        historical["date"].max().date(),
        len(historical),
        historical[TARGET].mean(),
        historical[TARGET].median(),
        historical[TARGET].min(),
        historical[TARGET].max(),
        historical[TARGET].iloc[-1],
        historical[TARGET].std(),
        historical[TARGET].iloc[-1] - historical[TARGET].iloc[0]
    ]
})

# Detect project output files
csv_files = sorted(OUTPUT_DIR.rglob("*.csv")) if OUTPUT_DIR.exists() else []

output_inventory = []
loaded_outputs = {}

for path in csv_files:
    try:
        data = pd.read_csv(path)
        loaded_outputs[path.name] = data
        output_inventory.append({
            "File": str(path.relative_to(PROJECT_ROOT)),
            "Rows": len(data),
            "Columns": len(data.columns),
            "Column names": ", ".join(map(str, data.columns[:12]))
        })
    except Exception as exc:
        output_inventory.append({
            "File": str(path.relative_to(PROJECT_ROOT)),
            "Rows": "Unreadable",
            "Columns": "Unreadable",
            "Column names": str(exc)
        })

output_inventory = pd.DataFrame(output_inventory)

# Identify forecast-related files
forecast_files = {
    name: data for name, data in loaded_outputs.items()
    if any(word in name.lower() for word in
           ["forecast", "prediction", "walk_forward", "baseline"])
}

# Identify uncertainty-related files
uncertainty_files = {
    name: data for name, data in loaded_outputs.items()
    if any(word in name.lower() for word in
           ["uncertainty", "interval", "confidence", "prediction_interval"])
}

# Identify scenario-related files
scenario_files = {
    name: data for name, data in loaded_outputs.items()
    if "scenario" in name.lower()
}

# Identify capacity and surge risk files
risk_files = {
    name: data for name, data in loaded_outputs.items()
    if any(word in name.lower() for word in
           ["capacity", "breach", "surge", "resource_pressure", "peak_care"])
}

# Calculate forecast accuracy when actual and prediction columns are available
accuracy_rows = []

actual_candidates = [
    "actual", "actual_value", "y_true", "true", "observed",
    "actual_children_in_hhs_care"
]

prediction_candidates = [
    "predicted", "prediction", "forecast", "y_pred",
    "predicted_value", "predicted_children_in_hhs_care"
]

for filename, data in forecast_files.items():
    normalized = {
        str(col).lower().strip(): col for col in data.columns
    }

    actual_col = next(
        (normalized[x] for x in actual_candidates if x in normalized),
        None
    )

    prediction_col = next(
        (normalized[x] for x in prediction_candidates if x in normalized),
        None
    )

    if actual_col is None or prediction_col is None:
        continue

    actual = pd.to_numeric(data[actual_col], errors="coerce")
    predicted = pd.to_numeric(data[prediction_col], errors="coerce")

    valid = pd.DataFrame({
        "actual": actual,
        "predicted": predicted
    }).dropna()

    if valid.empty:
        continue

    error = valid["actual"] - valid["predicted"]
    denominator = (
        valid["actual"].abs() + valid["predicted"].abs()
    ).replace(0, np.nan)

    ss_total = (
        (valid["actual"] - valid["actual"].mean()) ** 2
    ).sum()

    r2 = (
        1 - (error ** 2).sum() / ss_total
        if ss_total > 0 else np.nan
    )

    accuracy_rows.append({
        "Source file": filename,
        "Observations": len(valid),
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "MAPE (%)": (
            error.abs()[valid["actual"] != 0]
            / valid.loc[valid["actual"] != 0, "actual"].abs()
        ).mean() * 100 if (valid["actual"] != 0).any() else np.nan,
        "sMAPE (%)": (
            2 * error.abs() / denominator
        ).mean() * 100,
        "R-squared": r2
    })

accuracy_results = pd.DataFrame(accuracy_rows)

# Summarize forecast horizon evidence from forecast outputs
horizon_rows = []

for filename, data in forecast_files.items():
    horizon_col = next(
        (col for col in data.columns
         if any(term in str(col).lower()
                for term in ["horizon", "forecast_step", "lead_time"])),
        None
    )

    if horizon_col is not None:
        values = sorted(
            pd.to_numeric(data[horizon_col], errors="coerce")
            .dropna().unique().tolist()
        )
        horizon_rows.append({
            "Source file": filename,
            "Horizon column": str(horizon_col),
            "Detected horizons": ", ".join(map(str, values))
        })
    else:
        horizon_rows.append({
            "Source file": filename,
            "Horizon column": "Not identified",
            "Detected horizons": "Not independently verifiable"
        })

horizon_results = pd.DataFrame(horizon_rows)

# Produce concise evidence summaries for risk and scenario files
def summarize_files(file_dictionary):
    summaries = []

    for filename, data in file_dictionary.items():
        summaries.append({
            "File": filename,
            "Rows": len(data),
            "Columns": ", ".join(map(str, data.columns)),
            "Sample evidence": data.head(5).to_dict(orient="records")
        })

    return summaries

risk_summaries = summarize_files(risk_files)
scenario_summaries = summarize_files(scenario_files)
uncertainty_summaries = summarize_files(uncertainty_files)

# Identify historical trends without assuming missing dates are zero
historical["year_month"] = historical["date"].dt.to_period("M").astype(str)

monthly_demand = (
    historical.groupby("year_month")[TARGET]
    .agg(["mean", "median", "min", "max", "count"])
    .reset_index()
)

monthly_demand.columns = [
    "Month", "Average care load", "Median care load",
    "Minimum", "Maximum", "Observations"
]

# Create evidence-based narrative
latest_value = historical[TARGET].iloc[-1]
average_value = historical[TARGET].mean()
maximum_value = historical[TARGET].max()
minimum_value = historical[TARGET].min()

executive_overview = (
    f"The analysis covers {len(historical)} valid historical observations "
    f"from {historical['date'].min().date()} to "
    f"{historical['date'].max().date()}. "
    f"The latest recorded HHS care load is {latest_value:,.2f}. "
    f"The historical average is {average_value:,.2f}, while the observed "
    f"range is {minimum_value:,.2f} to {maximum_value:,.2f}. "
    f"The project contains {len(forecast_files)} forecast-related output "
    f"files, {len(risk_files)} risk-related files, "
    f"{len(scenario_files)} scenario-related files, and "
    f"{len(uncertainty_files)} uncertainty-related files."
)

# Print analysis results
print("=" * 75)
print("GOVERNMENT EXECUTIVE SUMMARY — EVIDENCE ANALYSIS")
print("=" * 75)

print("\nEXECUTIVE OVERVIEW")
print(executive_overview)

print("\nHISTORICAL CARE DEMAND")
display(historical_summary)

print("\nMONTHLY HISTORICAL DEMAND")
display(monthly_demand.tail(24))

print("\nFORECAST ACCURACY")
if accuracy_results.empty:
    print("No forecast file with identifiable actual and predicted columns was found.")
else:
    display(accuracy_results)

print("\nFORECAST HORIZON EVIDENCE")
display(horizon_results)

print("\nCAPACITY AND SURGE RISK FILES")
if risk_files:
    for name, data in risk_files.items():
        print(f"\n{name}")
        display(data.head(10))
else:
    print("No capacity or surge risk files were detected.")

print("\nSCENARIO COMPARISON FILES")
if scenario_files:
    for name, data in scenario_files.items():
        print(f"\n{name}")
        display(data.head(10))
else:
    print("No scenario output files were detected.")

print("\nUNCERTAINTY FILES")
if uncertainty_files:
    for name, data in uncertainty_files.items():
        print(f"\n{name}")
        display(data.head(10))
else:
    print("No uncertainty output files were detected.")

print("\nOUTPUT INVENTORY")
display(output_inventory)

print("\nAnalysis completed.")


GOVERNMENT EXECUTIVE SUMMARY — EVIDENCE ANALYSIS

EXECUTIVE OVERVIEW
The analysis covers 720 valid historical observations from 2023-01-12 to 2025-12-21. The latest recorded HHS care load is 2,484.00. The historical average is 6,061.27, while the observed range is 1,972.00 to 11,516.00. The project contains 22 forecast-related output files, 10 risk-related files, 8 scenario-related files, and 15 uncertainty-related files.

HISTORICAL CARE DEMAND


,Indicator,Value
0,First observed date,2023-01-12
1,Last observed date,2025-12-21
2,Number of valid observations,720
3,Average children in HHS care,6061.275
4,Median children in HHS care,6406.5
5,Minimum observed care load,1972
6,Maximum observed care load,11516
7,Latest observed care load,2484
8,Standard deviation,2833.070109
9,First-to-last change,-4082



MONTHLY HISTORICAL DEMAND


,Month,Average care load,Median care load,Minimum,Maximum,Observations
12,2024-01,8851.045455,8181.0,7974,10870,22
13,2024-02,8325.200000,8392.0,7813,8724,20
14,2024-03,8382.666667,8390.0,7257,8902,21
15,2024-04,7027.136364,6993.5,6630,7431,22
16,2024-05,7490.285714,7494.0,7356,7589,21
17,2024-06,7407.523810,7433.0,6874,7741,21
18,2024-07,6286.333333,6239.0,5743,6825,21
19,2024-08,6004.263158,5995.0,5821,6250,19
20,2024-09,6213.409091,6239.5,5925,6448,22
21,2024-10,6032.857143,6040.0,5889,6149,21



FORECAST ACCURACY


,Source file,Observations,MAE,RMSE,MAPE (%),sMAPE (%),R-squared
0,baseline_predictions.csv,3900,404.215602,807.706571,15.890016,12.638823,-0.875254
1,combined_model_predictions.csv,32300,1335.081774,2207.642920,56.764663,32.527973,-8.449971
2,prediction_intervals.csv,2730,1365.752740,2306.380938,62.861639,34.150336,-222.090021
3,test_predictions.csv,2730,1365.752740,2306.380938,62.861639,34.150336,-222.090021
4,validation_predictions.csv,416,383.675481,758.371194,14.705853,11.691040,-0.063657
5,machine_learning_recursive_predictions.csv,416,3470.051321,3567.486872,152.124213,83.348999,-16.324095
6,machine_learning_test_predictions.csv,1040,4078.515208,4093.480641,187.643647,96.459258,-613.997561
7,machine_learning_validation_predictions.csv,1040,3696.764884,3932.895694,151.584692,80.312152,-12.044224
8,forecast_revisions.csv,2931,1623.609453,2534.479099,74.492986,39.995090,-274.911920
9,statistical_test_forecasts.csv,20,193.656638,308.227295,8.748618,7.853209,-2.501462


... (67 outputs left out)

,Partition,Forecast_Origin,Actual_Target_Date,Horizon,Model,Actual,Predicted,Absolute_Error,Period_Type,Residual,...,Predicted_Surge,Actual_Surge,Actual_Regime,Capacity_Utilization,Predicted_Capacity_Breach,Previous_Forecast,Forecast_Revision,Absolute_Forecast_Revision,Interval_Method,Residual_Sample_Size
0,validation,2025-01-02,2025-01-05,1,Naive_Persistence,5882.0,6292.000000,410.000000,Unknown,-410.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
1,validation,2025-01-02,2025-01-05,1,Seasonal_Naive,5882.0,6622.000000,740.000000,Unknown,-740.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
2,validation,2025-01-02,2025-01-05,1,Moving_Average_7,5882.0,6519.142857,637.142857,Unknown,-637.142857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
3,validation,2025-01-02,2025-01-05,1,Moving_Average_14,5882.0,6524.500000,642.500000,Unknown,-642.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
4,validation,2025-01-02,2025-01-05,1,Moving_Average_30,5882.0,6292.933333,410.933333,Unknown,-410.933333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
5,validation,2025-01-02,2025-01-13,1,Naive_Persistence,5019.0,6292.000000,1273.000000,Unknown,-1273.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
6,validation,2025-01-02,2025-01-13,1,Seasonal_Naive,5019.0,6622.000000,1603.000000,Unknown,-1603.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
7,validation,2025-01-02,2025-01-13,1,Moving_Average_7,5019.0,6519.142857,1500.142857,Unknown,-1500.142857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
8,validation,2025-01-02,2025-01-13,1,Moving_Average_14,5019.0,6524.500000,1505.500000,Unknown,-1505.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416
9,validation,2025-01-02,2025-01-13,1,Moving_Average_30,5019.0,6292.933333,1273.933333,Unknown,-1273.933333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Model_Horizon_Residual_Bootstrap,416



OUTPUT INVENTORY


,File,Rows,Columns,Column names
0,outputs\annual_mean_patterns.csv,3,6,"date, children_apprehended_and_placed_in_cbp_c..."
1,outputs\annual_observation_counts.csv,3,6,"date, children_apprehended_and_placed_in_cbp_c..."
2,outputs\annual_observed_flow_sums.csv,3,4,"date, children_apprehended_and_placed_in_cbp_c..."
3,outputs\arima_candidate_parameters.csv,9,3,"Order, AIC, BIC"
4,outputs\baseline_predictions.csv,3900,8,"Dataset, Forecast_Origin, Actual_Target_Date, ..."
...,...,...,...,...
204,outputs\validation_predictions.csv,416,6,"Forecast_Origin, Actual_Target_Date, Horizon, ..."
205,outputs\variable_correlations.csv,5,6,"Unnamed: 0, children_apprehended_and_placed_in..."
206,outputs\walk_forward_predictions.csv,416,6,"Forecast_Origin, Actual_Target_Date, Horizon, ..."
207,outputs\weekly_observation_counts.csv,154,6,"date, children_apprehended_and_placed_in_cbp_c..."



Analysis completed.


## Document Export
After the analysis cell finishes successfully, run the export cell to generate the government executive summary in DOCX and PDF.

In [2]:

# Prepare and Export the Government Executive Summary

import sys
import subprocess
import importlib.util
from pathlib import Path
import pandas as pd
import numpy as np
import re
from xml.sax.saxutils import escape

# Install document-generation packages in the active kernel
for module_name, package_name in [
    ("docx", "python-docx"),
    ("reportlab", "reportlab")
]:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package_name
        ])

from docx import Document
from docx.shared import Inches, Pt
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak
)
from reportlab.lib.units import inch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORT_DIR = PROJECT_ROOT / "government_executive_summary"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Verify that the analysis cell has run
required_variables = [
    "historical_summary", "monthly_demand", "accuracy_results",
    "horizon_results", "output_inventory", "executive_overview",
    "risk_files", "scenario_files", "uncertainty_files"
]

missing = [name for name in required_variables if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the analysis cell before exporting. Missing: "
        + ", ".join(missing)
    )

# Prepare the government-facing narrative
sections = [
    ("1. Executive Overview", executive_overview),

    ("2. Historical Care Demand",
     "Historical demand is summarized using observed HHS care-load "
     "records. The report distinguishes observed values from estimates. "
     "Missing reporting dates are not interpreted as zero demand."),

    ("3. Forecast Horizons",
     "The project evaluates 1-day, 7-day, 14-day, and 30-day forecast "
     "horizons. The detected forecast files and horizon metadata are "
     "presented in the evidence tables. A horizon is not claimed as "
     "validated unless supported by an output file."),

    ("4. Forecast Accuracy",
     "Forecast accuracy is reported only where output files contain "
     "identifiable actual and predicted observations. MAE, RMSE, MAPE, "
     "sMAPE, and R-squared are calculated from matching observations. "
     "These metrics should be interpreted in the context of the "
     "validation and test periods."),

    ("5. Surge Warnings",
     "Surge warnings should be interpreted using the project's generated "
     "risk and forecast outputs. A warning threshold must be supported "
     "by the project's documented assumptions; no threshold is invented "
     "in this report."),

    ("6. Capacity Breach Risks",
     "Capacity breach risks are based on available capacity, breach "
     "probability, and operational resource-pressure outputs. Where "
     "capacity thresholds or probabilities are unavailable, the report "
     "identifies the evidence gap instead of asserting a breach."),

    ("7. Scenario Comparison",
     "Scenario comparisons are presented from the generated scenario "
     "outputs. Scenario assumptions should be reviewed before using "
     "scenario projections for policy or resource allocation decisions."),

    ("8. Operational Implications",
     "Forecasts can support staffing, placement planning, resource "
     "readiness, monitoring frequency, and contingency preparation. "
     "Forecasts are decision-support evidence and do not replace "
     "operational verification or statutory decision-making."),

    ("9. Government Recommendations",
     "Maintain regular monitoring of observed care load; review short- "
     "and medium-term forecasts together; define and document capacity "
     "thresholds; prepare escalation procedures for credible surge "
     "signals; compare scenario assumptions before allocating resources; "
     "and update forecasts as new observations become available."),

    ("10. Limitations and Uncertainty",
     "Forecasts are subject to historical data coverage, reporting "
     "irregularity, model error, structural changes, and uncertainty "
     "in scenario assumptions. Missing dates are not treated as zero. "
     "Prediction intervals and scenario ranges should be considered "
     "when interpreting point forecasts. Forecasts should not be treated "
     "as guarantees of future demand.")
]

# Build DOCX
doc = Document()
section = doc.sections[0]
section.top_margin = Inches(0.7)
section.bottom_margin = Inches(0.7)
section.left_margin = Inches(0.8)
section.right_margin = Inches(0.8)

doc.styles["Normal"].font.name = "Arial"
doc.styles["Normal"].font.size = Pt(10)

title = doc.add_heading(
    "Government Executive Summary", level=0
)
title.alignment = 1

doc.add_paragraph(
    "Predictive Forecasting of Care Load & Placement Demand"
)
doc.add_paragraph(
    "Government decision-support report"
)

doc.add_heading("Evidence-Based Executive Findings", level=1)

for heading, paragraph in sections:
    doc.add_heading(heading, level=1)
    doc.add_paragraph(paragraph)

def add_docx_table(document, dataframe, heading):
    document.add_heading(heading, level=2)

    if dataframe is None or dataframe.empty:
        document.add_paragraph(
            "No corresponding evidence was available in the generated outputs."
        )
        return

    display_df = dataframe.copy()
    display_df = display_df.replace({np.nan: "Not available"})
    display_df = display_df.astype(str)

    table = document.add_table(rows=1, cols=len(display_df.columns))
    table.style = "Table Grid"

    for i, col in enumerate(display_df.columns):
        table.rows[0].cells[i].text = str(col)

    for _, row in display_df.iterrows():
        cells = table.add_row().cells
        for i, value in enumerate(row):
            cells[i].text = str(value)

add_docx_table(doc, historical_summary, "Historical Demand Indicators")
add_docx_table(doc, monthly_demand.tail(24), "Recent Monthly Demand")
add_docx_table(doc, horizon_results, "Forecast Horizon Evidence")
add_docx_table(doc, accuracy_results, "Forecast Accuracy Results")
add_docx_table(doc, output_inventory, "Generated Output Inventory")

doc.add_heading("11. Surge and Capacity Risk Evidence", level=1)

if risk_files:
    for filename, data in risk_files.items():
        doc.add_heading(filename, level=2)
        add_docx_table(doc, data.head(20), "Available risk evidence")
else:
    doc.add_paragraph(
        "No risk-related output files were detected. "
        "Surge and capacity conclusions therefore require further evidence."
    )

doc.add_heading("12. Scenario Comparison Evidence", level=1)

if scenario_files:
    for filename, data in scenario_files.items():
        doc.add_heading(filename, level=2)
        add_docx_table(doc, data.head(20), "Scenario output")
else:
    doc.add_paragraph(
        "No scenario comparison output files were detected."
    )

doc.add_heading("13. Uncertainty Evidence", level=1)

if uncertainty_files:
    for filename, data in uncertainty_files.items():
        doc.add_heading(filename, level=2)
        add_docx_table(doc, data.head(20), "Uncertainty output")
else:
    doc.add_paragraph(
        "No uncertainty output files were detected."
    )

doc.add_paragraph(
    "Prepared from the project's available historical dataset and "
    "generated analytical outputs. Numerical conclusions are limited "
    "to the evidence detected at the time of report generation."
)

DOCX_PATH = REPORT_DIR / "Government_Executive_Summary.docx"
doc.save(DOCX_PATH)

# Build PDF
styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="GovTitle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=18,
    leading=23,
    alignment=TA_CENTER,
    spaceAfter=16
))

styles.add(ParagraphStyle(
    name="GovBody",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=9,
    leading=13,
    alignment=TA_JUSTIFY,
    spaceAfter=7
))

styles.add(ParagraphStyle(
    name="GovHeading",
    parent=styles["Heading1"],
    fontName="Helvetica-Bold",
    fontSize=13,
    leading=17,
    spaceBefore=12,
    spaceAfter=7
))

story = [
    Paragraph("Government Executive Summary", styles["GovTitle"]),
    Paragraph(
        "Predictive Forecasting of Care Load &amp; Placement Demand",
        styles["GovBody"]
    ),
    Spacer(1, 12)
]

for heading, paragraph in sections:
    story.append(Paragraph(escape(heading), styles["GovHeading"]))
    story.append(Paragraph(escape(paragraph), styles["GovBody"]))

def add_pdf_table(story_list, dataframe, heading):
    story_list.append(Paragraph(escape(heading), styles["GovHeading"]))

    if dataframe is None or dataframe.empty:
        story_list.append(Paragraph(
            "No corresponding evidence was available.",
            styles["GovBody"]
        ))
        return

    table_df = dataframe.copy().replace({np.nan: "Not available"})
    table_data = [
        [str(x) for x in table_df.columns]
    ]

    for row in table_df.astype(str).values.tolist():
        table_data.append([
            value[:100] for value in row
        ])

    table = Table(table_data, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 7),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("PADDING", (0, 0), (-1, -1), 4)
    ]))
    story_list.append(table)
    story_list.append(Spacer(1, 8))

add_pdf_table(story, historical_summary, "Historical Demand Indicators")
add_pdf_table(story, monthly_demand.tail(24), "Recent Monthly Demand")
add_pdf_table(story, horizon_results, "Forecast Horizon Evidence")
add_pdf_table(story, accuracy_results, "Forecast Accuracy Results")
add_pdf_table(story, output_inventory, "Generated Output Inventory")

for heading, collection in [
    ("Surge and Capacity Risk Evidence", risk_files),
    ("Scenario Comparison Evidence", scenario_files),
    ("Uncertainty Evidence", uncertainty_files)
]:
    story.append(Paragraph(heading, styles["GovHeading"]))

    if collection:
        for filename, data in collection.items():
            add_pdf_table(story, data.head(20), filename)
    else:
        story.append(Paragraph(
            "No corresponding output files were detected.",
            styles["GovBody"]
        ))

PDF_PATH = REPORT_DIR / "Government_Executive_Summary.pdf"

pdf = SimpleDocTemplate(
    str(PDF_PATH),
    pagesize=A4,
    rightMargin=0.55 * inch,
    leftMargin=0.55 * inch,
    topMargin=0.55 * inch,
    bottomMargin=0.55 * inch
)

pdf.build(story)

# Verify export
print("=" * 70)
print("GOVERNMENT EXECUTIVE SUMMARY EXPORT")
print("=" * 70)

for path in [DOCX_PATH, PDF_PATH]:
    if path.exists() and path.stat().st_size > 0:
        print(f"SUCCESS: {path}")
        print(f"Size: {path.stat().st_size:,} bytes")
    else:
        print(f"FAILED: {path}")

print("=" * 70)


GOVERNMENT EXECUTIVE SUMMARY EXPORT
SUCCESS: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\government_executive_summary\Government_Executive_Summary.docx
Size: 80,636 bytes
SUCCESS: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\government_executive_summary\Government_Executive_Summary.pdf
Size: 127,010 bytes
